# Financial NER — Exploratory Data Analysis & Model Evaluation

This notebook covers:
1. Dataset loading and EDA
2. Label distribution visualisation
3. Feature engineering inspection
4. Loading the trained model and running predictions
5. Error analysis

In [ ]:
import os, sys, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

# Make project root importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

plt.style.use('dark_background')
COLORS = ['#3b82f6','#a78bfa','#fbbf24','#34d399','#f87171']

print('Setup complete ✓')

## 1  Load Dataset

In [ ]:
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'sample_dataset.csv')
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Sentences: {df["sentence_id"].nunique()}')
df.head(15)

## 2  Label Distribution

In [ ]:
label_counts = df['label'].value_counts()
print(label_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart
ax = axes[0]
bars = ax.bar(label_counts.index, label_counts.values,
              color=COLORS[:len(label_counts)], edgecolor='none', alpha=.85)
ax.set_title('Token count per label', fontsize=12)
ax.bar_label(bars, padding=3)
ax.set_xlabel('Label')
ax.set_ylabel('Count')

# Pie chart (entities only)
ax2 = axes[1]
ent_counts = label_counts.drop('O', errors='ignore')
ax2.pie(ent_counts.values, labels=ent_counts.index,
        colors=COLORS[:len(ent_counts)], autopct='%1.1f%%',
        startangle=140)
ax2.set_title('Entity label share (O excluded)', fontsize=12)

plt.tight_layout()
plt.show()

## 3  Sentence-level Statistics

In [ ]:
sent_stats = df.groupby('sentence_id').agg(
    num_tokens=('token', 'count'),
    num_entities=('label', lambda x: (x != 'O').sum())
)
sent_stats['entity_density'] = sent_stats['num_entities'] / sent_stats['num_tokens']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, c in zip(axes, ['num_tokens', 'num_entities', 'entity_density'], COLORS):
    ax.hist(sent_stats[col], bins=10, color=c, edgecolor='none', alpha=.8)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('Value')
    ax.set_ylabel('Sentences')
plt.tight_layout()
plt.show()

print(sent_stats.describe().round(2))

## 4  Feature Engineering Inspection

In [ ]:
from utils.preprocessing import build_token_features, build_feature_sequence

sample_tokens = ['Apple', 'AAPL', 'reported', '$', '97', 'billion', 'merger', 'GDP']
print('Token Feature Strings\n' + '='*50)
for tok in sample_tokens:
    print(f'{tok:12s} → {build_token_features(tok)}')

## 5  Load Best Model & Predict

In [ ]:
MODEL_PATH = os.path.join(PROJECT_ROOT, 'models', 'best_model.pkl')
META_PATH  = os.path.join(PROJECT_ROOT, 'models', 'model_meta.json')

if not os.path.exists(MODEL_PATH):
    print('Model not found. Run: python models/train.py')
else:
    with open(MODEL_PATH, 'rb') as fh:
        pipeline = pickle.load(fh)
    with open(META_PATH) as fh:
        meta = json.load(fh)
    print(f'Loaded: {meta["best_model"]}  F1={meta["weighted_f1"]}')

In [ ]:
from utils.preprocessing import (
    tokenise, build_feature_sequence, aggregate_entities, tokens_to_char_offsets
)
from utils.entity_utils import LABEL_META

TEST_SENTENCES = [
    'Apple Inc (AAPL) reported record earnings of $97 billion this quarter.',
    'The Federal Reserve raised interest rates by 25 basis points, causing the S&P 500 to drop.',
    'Tesla TSLA completed a merger with SolarCity worth $2.6 billion.',
    'Bitcoin BTC crossed $50,000 after SEC approved a Bitcoin ETF.',
    'NVIDIA NVDA announced a 4-for-1 stock split sending shares to all-time highs.',
]

for sent in TEST_SENTENCES:
    tokens = tokenise(sent)
    feats  = build_feature_sequence(tokens)
    labels = pipeline.predict(feats)
    entities = aggregate_entities(tokens, labels)
    ents_only = [e for e in entities if e['label'] != 'O']
    print(f'\n» {sent}')
    for e in ents_only:
        icon = LABEL_META[e['label']]['icon']
        print(f'  {icon} [{e["label"]:10s}] {e["text"]}')

## 6  Model Comparison Charts

In [ ]:
import glob
REPORT_DIR = os.path.join(PROJECT_ROOT, 'models', 'reports')
imgs = glob.glob(os.path.join(REPORT_DIR, '*.png'))

if not imgs:
    print('No report images found. Run train.py first.')
else:
    from IPython.display import display, Image
    for img_path in sorted(imgs):
        print(os.path.basename(img_path))
        display(Image(filename=img_path, width=700))

## 7  Error Analysis

In [ ]:
from sklearn.model_selection import train_test_split

X_all, y_all = [], []
for _, grp in df.groupby('sentence_id'):
    tokens = grp['token'].tolist()
    labels_list = grp['label'].tolist()
    feats  = build_feature_sequence(tokens)
    X_all.extend(feats)
    y_all.extend(labels_list)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

y_pred = pipeline.predict(X_test)

# Find misclassified tokens
errors = [(xt, yt, yp) for xt, yt, yp in zip(X_test, y_test, y_pred) if yt != yp]
print(f'Total errors: {len(errors)} / {len(y_test)}  ({100*len(errors)/len(y_test):.1f}%)')
print('\nSample misclassifications (feat | true | pred):')
for feat, true, pred in errors[:15]:
    tok = feat.split()[0]  # first word is lowercased token
    print(f'  {tok:15s}  true={true:10s}  pred={pred}')